# 07 pamoka – Planavimo dizaino šablonas

Šiame užrašų knygelėje demonstruojamas **Planavimo dizaino šablonas** AI agentams, naudojant Microsoft Agent Framework. 
Išmoksite, kaip sudėtingas kelionių užklausas suskaidyti į struktūruotas tarpines užduotis, priskirti jas specialistų agentams 
ir įvykdyti galutinį planą – visa tai naudojant struktūruotą išvestį, paremta Pydantic modeliais.


## Sąranga


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Užduoties skaidymas

Užduoties skaidymas yra pagrindinis planavimo dizaino šablonas. Vietoj to, kad vienas agentas tvarkytų sudėtingą užklausą nuo pradžios iki pabaigos, mes problemą išskaidome į mažesnes, aiškiai apibrėžtas **použduotis**. Kiekviena požduotis priskiriama specializuotam agentui (pvz., skrydžiai, viešbučiai, veiklos) su aiškiomis prioriteto ir priklausomybės tvarkomis.

Šis požiūris suteikia keletą privalumų:
- **Aiškumas**: kiekviena požduotis turi vieną atsakomybę.
- **Lygiagreti vykdymas**: nepriklausomos požduotys gali vykdyti lygiagrečiai.
- **Patikimumas**: klaidos izoliuojamos atskirose požduotyse.
- **Biudžeto sekimas**: kaštai apskaičiuojami kiekvienai požduočiai atskirai ir apibendrinami.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Sukuriant planavimo agentą su struktūruotu išvedimu

Planavimo agentas veikia kaip **registratūros koordinatorius**. Gavęs aukšto lygio kelionės užklausą, jis sukuria struktūruotą `TravelPlan` — išskaido užklausą į poskyrius, nustato prioritetus ir identifikuoja priklausomybes, kad konsjeržas ar vykdymo sluoksnis galėtų atlikti darbą.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Plano vykdymas naudojant specialistų įrankius

Kai registratūros agentas paruošia struktūruotą planą, jį įgyvendina **konsjeržo agentas**.  
Kiekvienas specialistas įrankis tvarko vieną užduočių kategoriją (skrydžius, viešbučius, veiklas). Konsjeržas vykdo plano užduočių tvarką pagal priklausomybę ir siunčia kiekvieną užduotį tinkamam įrankiui.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Santrauka

Šiame pamokoje išmokote **Planavimo dizaino šabloną** dirbtinio intelekto agentams:

1. **Užduoties skaidymas** — Priekinio stalo planavimo agentas suskaido sudėtingą kelionės užklausą į struktūrizuotas po-užduotis naudodamas Pydantic modelius, priskirdamas kiekvieną specialisto agentui su prioritetais ir priklausomybėmis.
2. **Struktūruotas išvestis** — Perdavinėjant `response_format`, agentas grąžina patvirtintą `TravelPlan` objektą vietoje laisvos formos teksto, todėl tolimesnis apdorojimas yra patikimas.
3. **Plano vykdymas** — Konsjeržo agentas iteruoja per po-užduotis naudodamas specializuotus įrankius (`book_flight`, `reserve_hotel`, `book_activity`), kad įgyvendintų planą ir praneštų apie rezultatus.

Šis šablonas atskiria *ką daryti* (planavimą) nuo *kaip daryti* (vykdymo), todėl agentai yra moduliškesni, lengviau testuojami ir plečiami.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Atsakomybės apribojimas**:
Šis dokumentas buvo išverstas naudojant dirbtinio intelekto vertimo paslaugą [Co-op Translator](https://github.com/Azure/co-op-translator). Nors siekiame tikslumo, prašome atkreipti dėmesį, kad automatizuoti vertimai gali turėti klaidų ar netikslumų. Originalus dokumentas jo gimtąja kalba turėtų būti laikomas autoritetingu šaltiniu. Dėl svarbios informacijos rekomenduojama pasinaudoti profesionalaus žmogaus vertimu. Mes neatsakome už jokius nesusipratimus ar neteisingus aiškinimus, kylančius dėl šio vertimo naudojimo.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
